## はじめに（4時間パートナー版）

このノートブックでは、GlacierStyle ECサイトの各種データをAI機能で加工・変換します。

**前提条件:**
- part1_data_ingest_4h.ipynb が実行済みであること

**処理内容:**
1. SNSログの分類・感情分析（AI_EXTRACT / AI_SENTIMENT / AI_CLASSIFY）
2. 音声ログの要約・分類・マスキング（AI_REDACT / AI_CLASSIFY / AI_AGG）
3. 商品マスタとの突合（AI_SIMILARITY）

> **Note:** 広告クリエイティブ分析とドキュメントチャンク化は setup_4h.sql で事前構築済みです。

**所要時間目安:** 約45分

In [104]:
%%sql -r result_env_setup
-- ============================================================================
-- 環境設定
-- ============================================================================
-- 使用するウェアハウスとスキーマを設定
USE WAREHOUSE GLACIERSTYLE_WH;
USE SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

images_dir = 'images/part3/'

def display_image(image_file: str) -> None:
    image_path = os.path.join(images_dir, image_file)
    img = Image.open(image_path)
    plt.figure(figsize=(15, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

In [ ]:
display_image('architecture.png')

## 3. データの加工・変換

このセクションでは、Raw層のデータをAI機能を活用して加工・変換し、Silver/Gold層のテーブルを作成します。

### 3-1. SNSログの分類・感情分析

SNSメンション（Twitter/Instagram等）の生データに対して、以下のAI処理を実施:
- **AI_EXTRACT**: 商品名・カテゴリ・問い合わせタイプの抽出
- **AI_SENTIMENT**: 投稿の感情分析（ポジティブ/ネガティブ/ニュートラル）
- **AI_CLASSIFY**: 投稿カテゴリの分類（称賛/クレーム/質問/提案）

In [ ]:
display_image('ai_extract.png')

In [ ]:
display_image('ai_sentiment.png')

In [ ]:
display_image('ai_classify.png')

In [106]:
%%sql -r result_silver_sns_analyzed
-- ============================================================================
-- SNS生ログのAI分析とGold層への保存
-- ============================================================================
-- Gold層テーブルの作成
CREATE OR REPLACE TABLE gold_sns_mentions_analyzed AS
WITH extracted_data AS (
    SELECT 
        post_id,
        platform,
        post_type,
        username,
        display_name,
        content,
        posted_at,
        likes,
        retweets,
        replies,
        hashtags,
        mentioned_products,
        media_urls,
        
        SNOWFLAKE.CORTEX.AI_EXTRACT(
            content,
            OBJECT_CONSTRUCT(
                'product_name', 'mentioned product name',
                'category', 'product category (e.g., ファッション, インテリア, テック)',
                'inquiry_type', 'inquiry type (e.g., 質問, レビュー, クレーム, 称賛, 提案)'
            )
        ) AS extracted_info
        
    FROM raw_sns_mentions
),
sentiment_analysis AS (
    SELECT 
        *,
        SNOWFLAKE.CORTEX.AI_SENTIMENT(content) AS sentiment_result
    FROM extracted_data
),
classified_data AS (
    SELECT 
        *,
        SNOWFLAKE.CORTEX.AI_CLASSIFY(
            content,
            ['称賛', 'クレーム', '質問', '提案']
        ) AS classification_result
    FROM sentiment_analysis
)
SELECT 
    post_id,
    platform,
    post_type,
    username,
    display_name,
    content,
    posted_at,
    likes,
    retweets,
    replies,
    hashtags,
    mentioned_products,
    media_urls,
    
    -- AI抽出情報
    extracted_info:response.product_name::VARCHAR AS extracted_product_name,
    extracted_info:response.category::VARCHAR AS extracted_category,
    extracted_info:response.inquiry_type::VARCHAR AS inquiry_type,
    
    -- 感情分析結果
    sentiment_result:categories[0].name::VARCHAR AS overall_sentiment,
    sentiment_result:categories[0].sentiment::VARCHAR AS sentiment,
    
    -- カテゴリ分類結果
    classification_result:labels[0]::VARCHAR AS post_category,

    -- メタデータ
    CURRENT_TIMESTAMP() AS processed_at
    
FROM classified_data;

In [107]:
%%sql -r preview_silver_sns
-- ============================================================================
-- 分析結果の確認
-- ============================================================================
-- Gold層テーブルの内容をプレビュー
SELECT * FROM gold_sns_mentions_analyzed LIMIT 10;

### 3-2. 音声ログの要約・分類・マスキング

コールセンター音声ログの文字起こしデータに対して、以下のAI処理を実施:
- **AI_REDACT**: 個人情報（氏名・電話番号・住所・クレカ番号）の自動マスキング
- **AI_SENTIMENT**: 顧客感情の分析
- **AI_CLASSIFY**: 問い合わせカテゴリ分類
- **AI_AGG**: 通話内容の要約生成

In [ ]:
display_image('ai_redact.png')

In [ ]:
display_image('ai_agg.png')

In [ ]:
display_image('ai_agg_2.png')

In [108]:
%%sql -r result_agg_voc_summary
-- ============================================================================
-- 音声ログのAI分析とGold層への保存
-- ============================================================================
CREATE OR REPLACE TABLE gold_voice_logs AS
SELECT
    -- 元のカラム（transcribed_text以外）
    * EXCLUDE transcribed_text,
    
    -- 1. AI_REDACTによる個人情報のマスキング
    -- 氏名、電話番号、住所、クレジットカード番号を自動検出・マスキング
    SNOWFLAKE.CORTEX.AI_REDACT(
        transcribed_text
    ) AS transcribed_text_masked,
    
    -- 2. AI_SENTIMENTによる顧客感情の分析
    -- マスキング済みテキストに対して感情分析を実施
    SNOWFLAKE.CORTEX.AI_SENTIMENT(
        transcribed_text_masked
    ) AS sentiment_result,
    
    -- 感情の詳細を抽出
    sentiment_result:categories[0].name::VARCHAR AS overall_sentiment,
    sentiment_result:categories[0].sentiment::VARCHAR AS sentiment,
    
    -- 3. AI_CLASSIFYによる問い合わせカテゴリ分類
    SNOWFLAKE.CORTEX.AI_CLASSIFY(
        transcribed_text_masked,
        ['商品に関する問い合わせ', '配送に関する問い合わせ', '返品・交換', '決済・支払い', 'アカウント・会員登録', 'クレーム', 'その他']
    ) AS classification_result,
    
    -- 分類の詳細を抽出
    classification_result:labels[0]::VARCHAR AS inquiry_category,
    
    -- 4. AI_AGGによる通話内容の要約生成（GROUP BY対象）
    AI_AGG(
        transcribed_text_masked, 
        '音声ログを400文字以内で要約してください。顧客の主な問い合わせ内容、要望、および解決状況を含めてください。'
    ) AS transcribed_text_summary,
    
    -- メタデータ
    CURRENT_TIMESTAMP() AS processed_at

FROM raw_voice_logs
GROUP BY ALL;

In [ ]:
%%sql -r dataframe_3
-- ============================================================================
-- 音声ログGold層の確認
-- ============================================================================
SELECT * FROM gold_voice_logs LIMIT 10;

### 3-3〜3-5: 広告分析・ドキュメント処理（事前構築済み）

以下のGold層テーブルは `setup_4h.sql` のバックアップ復元で構築済みです：

| テーブル名 | 内容 | 使用AI関数 |
|-----------|------|------------|
| gold_ad_creative_analysis | 広告クリエイティブ分析 | AI_COMPLETE（マルチモーダル） |
| gold_faq_documents | FAQドキュメント（チャンク化済み） | SPLIT_TEXT_MARKDOWN_HEADER |
| gold_operation_manuals | 運用マニュアル（チャンク化済み） | SPLIT_TEXT_MARKDOWN_HEADER |

> 通常版（6パート版）では、これらのテーブルもPart3で構築します。
> 興味のある方は `part3_data_process.ipynb` のセクション3-3〜3-5を参照してください。

### 3-6. 商品マスタとの突合（名寄せ）

SNSメンションから抽出した商品名と商品マスタをAI_SIMILARITYで突合:
- カテゴリでフィルタ後、商品名の類似度を計算
- 類似度が高い順にソート

In [ ]:
display_image('ai_similarity.png')

In [ ]:
display_image('embedding.png')

In [122]:
%%sql -r result_product_matching
-- ============================================================================
-- SNSメンションと商品マスタの突合（名寄せ）
-- ============================================================================
-- AI_SIMILARITYを使用して商品名の類似度を計算
CREATE OR REPLACE TABLE gold_sns_mentions_with_product_master AS
WITH product_match AS (
    SELECT 
        *
    FROM gold_sns_mentions_analyzed t1
    INNER JOIN dim_products t2
        ON t2.category_l1 = t1.extracted_category
)
SELECT
    *,
    AI_SIMILARITY(extracted_product_name, product_name) AS similarity
FROM product_match
ORDER BY post_id, similarity DESC;

In [124]:
%%sql -r dataframe_1
-- ============================================================================
-- 商品マスタ突合結果の確認
-- ============================================================================
SELECT * FROM gold_sns_mentions_with_product_master ORDER BY post_id LIMIT 1000;

## まとめ

このノートブックでは、Snowflake Cortex AI機能を活用してRaw層データを加工・変換しました。

### 作成したGold層テーブル

| テーブル名 | 処理内容 | 使用AI関数 |
|-----------|---------|------------|
| gold_sns_mentions_analyzed | SNS分類・感情分析 | AI_EXTRACT, AI_SENTIMENT, AI_CLASSIFY |
| gold_voice_logs | 音声ログ要約・マスキング | AI_REDACT, AI_CLASSIFY, AI_AGG |
| gold_sns_mentions_with_product_master | SNS×商品マスタ突合 | AI_SIMILARITY |

### 事前構築済みテーブル（setup_4h.sqlで復元）

| テーブル名 | 処理内容 |
|-----------|----------|
| gold_ad_creative_analysis | 広告クリエイティブ分析 |
| gold_faq_documents | FAQドキュメント |
| gold_operation_manuals | 運用マニュアル |

### 次のステップ

- **Part 5**: Cortex Search Serviceの作成と検索テスト